# Recess Ready: Philadelphia

**Question:** each evening, can we predict whether tomorrow will be unsafe for outdoor recess at Philadelphia schools, and **why** (heat, cold, bad air, ozone smog, flash-flood rain, dangerous wind, thunderstorms, heavy snow, or ice)?

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

## Step 1: Download Philadelphia's daily weather and air quality

We ask Open-Meteo (free, no API key) for one row per day in Philadelphia:
- **Weather** (archive API): hottest and coldest "feels like" temperature, rain, snow, strongest wind gust, and a **weather code**. The weather code is a number that says the worst weather of the day, for example 95 = thunderstorm and 66 = freezing rain.
- **Air quality** (air-quality API): PM2.5 (tiny particles) and ozone, every hour. We turn the hours into one number per day.

Note: Open-Meteo's air quality numbers are **estimates from a computer model (CAMS)**, not readings from a sensor. We say this in our write-up.

In [ ]:
import pandas as pd
import requests

LAT, LON = 39.95, -75.16            # Philadelphia (Center City)
START, END = "2022-08-01", "2026-11-12"

# Weather: one row per day
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather = requests.get(weather_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "daily": "apparent_temperature_max,apparent_temperature_min,precipitation_sum,"
             "snowfall_sum,wind_gusts_10m_max,weather_code",
    "timezone": "America/New_York",
}).json()
weather_df = pd.DataFrame(weather["daily"])
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Air quality: one row per hour
air_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
air = requests.get(air_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "hourly": "pm2_5,ozone",
    "timezone": "America/New_York",
}).json()
air_df = pd.DataFrame(air["hourly"]).set_index("time")
air_df.index = pd.to_datetime(air_df.index)

# PM2.5: average of the day's 24 hours
pm_daily = air_df["pm2_5"].resample("D").mean()
# Ozone: the EPA uses 8-hour averages, so we take the worst 8-hour average of each day
ozone_daily = air_df["ozone"].rolling(8).mean().resample("D").max()
air_daily = pd.DataFrame({"pm2_5": pm_daily, "ozone_8h": ozone_daily}).reset_index()

# Put weather and air side by side, one row per day
df = weather_df.merge(air_daily, on="time")
print(df.shape)
df.head()

## Step 2: How many unsafe *school* days are there, and why?

A day is **unsafe** if any one of these happens. ⚠️ **Team to-do:** look up and cite a real source for each threshold (e.g. NWS, US EPA, or our school district's recess policy), and change the number if the source says something different.

| Warning | Rule |
|---|---|
| Too hot | "feels like" max ≥ 32 °C (≈90 °F, NWS "extreme caution") |
| Too cold | "feels like" min ≤ −7 °C (≈20 °F) |
| Bad air (smoke/particles) | daily PM2.5 > 35.4 µg/m³ (US EPA "Unhealthy for Sensitive Groups") |
| Ozone smog | worst 8-hour ozone > 137 µg/m³ (≈70 ppb, US EPA standard) |
| Flash flood / heavy rain | ≥ 25 mm (≈1 inch) of rain in the day |
| Dangerous wind (hurricanes, storms) | gusts ≥ 74 km/h (≈46 mph, NWS wind advisory level) |
| Thunderstorm / lightning | weather code 95, 96 or 99 |
| Heavy snow | ≥ 5 cm (≈2 inches) of snow |
| Ice / freezing rain | weather code 56, 57, 66 or 67 |

**Left out on purpose:** tornadoes (not in our data, very rare in Philadelphia; "dangerous wind" and "thunderstorm" cover the storms that bring them), earthquakes (nobody can predict them, and they aren't climate), and UV/sunburn (not in our weather data).

We only count **school days**: weekdays from September to June (no recess in July and August). We ignore holidays for now.

**Why this check comes first:** if a warning almost never happens, the model can't learn it. We'd drop it or change the threshold now, not on Sunday.

In [ ]:
df["too_hot"] = df["apparent_temperature_max"] >= 32
df["too_cold"] = df["apparent_temperature_min"] <= -7
df["bad_air"] = df["pm2_5"] > 35.4
df["ozone_smog"] = df["ozone_8h"] > 137
df["heavy_rain"] = df["precipitation_sum"] >= 25
df["high_wind"] = df["wind_gusts_10m_max"] >= 74
df["thunderstorm"] = df["weather_code"].isin([95, 96, 99])
df["heavy_snow"] = df["snowfall_sum"] >= 5
df["ice"] = df["weather_code"].isin([56, 57, 66, 67])

hazards = ["too_hot", "too_cold", "bad_air", "ozone_smog", "heavy_rain",
           "high_wind", "thunderstorm", "heavy_snow", "ice"]
df["unsafe"] = df[hazards].any(axis=1)   # True if ANY hazard happens

is_weekday = df["time"].dt.weekday < 5           # Monday=0 ... Friday=4
is_school_month = ~df["time"].dt.month.isin([7, 8])
school = df[is_weekday & is_school_month].copy()
school["year"] = school["time"].dt.year

print("School days:", len(school))
print("Unsafe school days:", school["unsafe"].sum())
print()
print("Unsafe school days per year, by reason:")
print(school.groupby("year")[hazards + ["unsafe"]].sum().T)   # .T flips the table so it fits

## Step 3: Turn each day into a question about *tomorrow*

Right now each row is one day. We want: "using **today's** weather and air, predict **tomorrow's** dangers."

- `features` = what we know today (the model's inputs).
- `shift(-1)` moves tomorrow's answers up one row, so each row holds today's inputs **and** tomorrow's answers side by side.
- We keep only rows where **tomorrow is a school day**.

**Leakage check:** the inputs are only from today. Nothing from tomorrow goes into `features`.

In [ ]:
df = df.sort_values("time").reset_index(drop=True)
df["month"] = df["time"].dt.month

features = ["apparent_temperature_max", "apparent_temperature_min", "precipitation_sum",
            "snowfall_sum", "wind_gusts_10m_max", "weather_code", "pm2_5", "ozone_8h", "month"]

# Tomorrow's answers, moved up one row
target_cols = []
for h in hazards:
    df[h + "_tomorrow"] = df[h].shift(-1)
    target_cols.append(h + "_tomorrow")

df["tomorrow"] = df["time"] + pd.Timedelta(days=1)
tomorrow_is_school = (df["tomorrow"].dt.weekday < 5) & ~df["tomorrow"].dt.month.isin([7, 8])

data = df[tomorrow_is_school].dropna(subset=features + target_cols).copy()
data[target_cols] = data[target_cols].astype(bool)
data["unsafe_tomorrow"] = data[target_cols].any(axis=1)
print("Rows we can learn from:", len(data))

## Step 4: Split by time (never shuffle)

- **Train** on every school day **before** September 2025.
- **Test** on the **2025-26 school year**, which the model never sees while learning.

This is like a real forecast: you only ever know the past.

In [ ]:
TEST_START = "2025-09-01"
train = data[data["tomorrow"] < TEST_START]
test = data[data["tomorrow"] >= TEST_START]

print("Train:", train["tomorrow"].min().date(), "to", train["tomorrow"].max().date(), "-", len(train), "days")
print("Test: ", test["tomorrow"].min().date(), "to", test["tomorrow"].max().date(), "-", len(test), "days")
print("Unsafe share in train:", round(train["unsafe_tomorrow"].mean(), 2))
print("Unsafe share in test: ", round(test["unsafe_tomorrow"].mean(), 2))

## Step 5: The baseline, "tomorrow will be the same as today"

The simplest possible guess: whatever danger happened today, guess it happens again tomorrow. Our model is only useful if it beats this.

In [ ]:
baseline_pred = test[hazards].values.astype(bool)   # today's dangers = guess for tomorrow

## Step 6: Train the model (a random forest)

A **random forest** is a group of decision trees. Each tree asks yes/no questions like "is today's 'feels like' max above 30 °C?" and the trees vote.

- We train **one forest that predicts all 9 dangers at once** (scikit-learn allows several answers per row).
- `class_weight="balanced"` tells the model that rare unsafe days matter, so it doesn't just say "safe" every time.
- `random_state=42` makes the result the same every time we run it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                               class_weight="balanced", random_state=42)
model.fit(train[features], train[target_cols])
model_pred = model.predict(test[features]).astype(bool)

## Step 7: Model vs. baseline

For each danger we count, on the test year:
- **caught**: dangerous days we warned about (higher is better)
- **missed**: dangerous days we did NOT warn about (the worst mistake for kids)
- **false alarms**: we warned but it was fine (annoying, less harmful)

⚠️ If accuracy is above 95%, be suspicious and look for leakage.

In [ ]:
y_true = test[target_cols].values.astype(bool)

def score(true, pred):
    return {"caught": int((true & pred).sum()),
            "missed": int((true & ~pred).sum()),
            "false alarms": int((~true & pred).sum())}

rows = []
for i, h in enumerate(hazards):
    b = score(y_true[:, i], baseline_pred[:, i])
    m = score(y_true[:, i], model_pred[:, i])
    rows.append({"danger": h, "real days": int(y_true[:, i].sum()),
                 "baseline caught": b["caught"], "baseline missed": b["missed"], "baseline false alarms": b["false alarms"],
                 "model caught": m["caught"], "model missed": m["missed"], "model false alarms": m["false alarms"]})
print(pd.DataFrame(rows).to_string(index=False))

# Overall: is tomorrow unsafe for ANY reason?
true_any = y_true.any(axis=1)
base_any = baseline_pred.any(axis=1)
model_any = model_pred.any(axis=1)
print()
for name, pred in [("Baseline", base_any), ("Model", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

## Step 7b: Too many false alarms, so make the model less jumpy

**What we saw on our first run:** the model missed only 4 unsafe days, but raised 100 false alarms. No principal would trust a tool that cancels recess most days.

**The fix:** for each danger, the model gives a **chance from 0 to 1**. Until now it warned whenever the chance was above 0.5. We'll pick a better **cutoff**:

1. Hold back the **2024-25 school year** from the training data as a practice year ("validation").
2. Train a model on the years before it, and try cutoffs from 0.05 to 0.95.
3. Keep the **lowest cutoff whose false alarms are no more than the baseline's** on that year. Lowest means it catches as many unsafe days as possible.
4. Use that cutoff on the real test year (2025-26) **once**.

**Why not pick the cutoff using the test year?** That would be cheating: we'd be tuning on the answers. The test year stays untouched until the end.

In [ ]:
import numpy as np

def danger_probs(m, X):
    # The model's chance (0 to 1) of each danger, one column per danger
    cols = []
    for p, classes in zip(m.predict_proba(X), m.classes_):
        classes = list(classes)
        cols.append(p[:, classes.index(True)] if True in classes else np.zeros(len(X)))
    return np.column_stack(cols)

VAL_START = "2024-09-01"
fit_part = train[train["tomorrow"] < VAL_START]
val_part = train[train["tomorrow"] >= VAL_START]

tuner = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                               class_weight="balanced", random_state=42)
tuner.fit(fit_part[features], fit_part[target_cols])

val_true = val_part[target_cols].values.astype(bool).any(axis=1)
val_base = val_part[hazards].values.astype(bool).any(axis=1)
val_prob = danger_probs(tuner, val_part[features]).max(axis=1)   # chance of the likeliest danger
base_false_alarms = int((~val_true & val_base).sum())

THRESHOLD = 0.95                                  # fallback: very cautious
for t in np.arange(0.05, 1.0, 0.05):
    if (~val_true & (val_prob >= t)).sum() <= base_false_alarms:
        THRESHOLD = round(float(t), 2)
        break
print("Baseline false alarms on the practice year:", base_false_alarms)
print("Chosen cutoff:", THRESHOLD)

# Use the cutoff on the real test year, with the model trained on all training years
model_pred = danger_probs(model, test[features]) >= THRESHOLD
model_any = model_pred.any(axis=1)
for name, pred in [("Baseline", base_any), ("Model (tuned)", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

From here on, every step uses the **tuned** model.

## Step 8: Confusion matrix (overall "unsafe tomorrow")

A 2x2 table: rows are what really happened, columns are what the model said.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(true_any, model_any, display_labels=["Safe", "Unsafe"], cmap="Blues")
plt.title("Recess Ready: model on the 2025-26 school year")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9: Where does it fail?

Real unsafe days the model **missed**. We look at a few and ask why.

In [ ]:
missed = test[true_any & ~model_any]
print("Missed unsafe days:", len(missed))
for _, row in missed.head(3).iterrows():
    real = [h for h in hazards if row[h + "_tomorrow"]]
    print(f"- {row['tomorrow'].date()}: really {real}. "
          f"Today's feels-like max {row['apparent_temperature_max']:.0f} C, min {row['apparent_temperature_min']:.0f} C, "
          f"PM2.5 {row['pm2_5']:.0f}, gusts {row['wind_gusts_10m_max']:.0f} km/h")

## Step 10: Does it work equally well in every season?

Our fairness check: the same accuracy and missed-day counts, split by season.

In [ ]:
def season(month):
    if month in [9, 10, 11]: return "Fall"
    if month in [12, 1, 2]: return "Winter"
    if month in [3, 4, 5]: return "Spring"
    return "June"

seasons = test["tomorrow"].dt.month.map(season).values
rows = []
for s in ["Fall", "Winter", "Spring", "June"]:
    keep = seasons == s
    t, m, b = true_any[keep], model_any[keep], base_any[keep]
    rows.append({"season": s, "days": int(keep.sum()), "unsafe days": int(t.sum()),
                 "model accuracy": round((m == t).mean(), 2),
                 "baseline accuracy": round((b == t).mean(), 2),
                 "model missed": int((t & ~m).sum())})
print(pd.DataFrame(rows).to_string(index=False))

## Step 11: The demo app

Pick a school day from the test year. The app shows what Recess Ready would have told the school **the evening before**, and what really happened.

How the app is built:
- `recess_check()` is one Python function. It takes a date, asks the model for a prediction, and returns a small web page (HTML) as text.
- `CSS` is the styling: colors, fonts and layout. The look is a school playground: asphalt, painted court lines, and a recess flag that turns green or red.
- `gr.Blocks` puts a dropdown, a button and the result on the page.

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

In [ ]:
import gradio as gr

NAMES = {"too_hot": ("🔥", "Too hot"), "too_cold": ("🥶", "Too cold"),
         "bad_air": ("😷", "Smoke / particles"), "ozone_smog": ("🌫️", "Ozone smog"),
         "heavy_rain": ("🌊", "Flash-flood rain"), "high_wind": ("🌀", "Dangerous wind"),
         "thunderstorm": ("⚡", "Lightning"), "heavy_snow": ("❄️", "Heavy snow"),
         "ice": ("🧊", "Ice")}

# The model's real record on the test year (calculated, never typed in by hand)
model_caught = int((true_any & model_any).sum())
base_caught = int((true_any & base_any).sum())
total_unsafe = int(true_any.sum())

test_dates = test["tomorrow"].dt.strftime("%Y-%m-%d").tolist()

def recess_check(date_str):
    idx = test.index[test["tomorrow"].dt.strftime("%Y-%m-%d") == date_str][0]
    pred = danger_probs(model, test.loc[[idx], features])[0] >= THRESHOLD
    warned = [h for h, p in zip(hazards, pred) if p]
    happened = [h for h in hazards if test.loc[idx, h + "_tomorrow"]]
    day = test.loc[idx, "tomorrow"].strftime("%A, %B %-d, %Y")

    if warned:
        flag, call = "red", "Plan indoor recess"
        why = "Watch out for: " + ", ".join(NAMES[h][1].lower() for h in warned) + "."
    else:
        flag, call = "green", "Outdoor recess"
        why = "No dangers expected."

    # The 3x3 "four-square court": one square per danger
    squares = ""
    for h in hazards:
        icon, name = NAMES[h]
        state = "on" if h in warned else "off"
        real = '<span class="real">happened</span>' if h in happened else ""
        squares += f'<div class="sq {state}"><span class="ic">{icon}</span><span class="nm">{name}</span>{real}</div>'

    if happened:
        truth = "What really happened: " + ", ".join(NAMES[h][1].lower() for h in happened) + "."
    else:
        truth = "What really happened: a safe day."
    if warned and happened and not set(warned) & set(happened):
        verdict = "Right that it was unsafe, but for the wrong reason."
    elif (len(warned) > 0) == (len(happened) > 0):
        verdict = "Recess Ready got this day right."
    elif happened:
        verdict = "Missed: Recess Ready said it was safe."
    else:
        verdict = "False alarm: the day was actually safe."

    return f"""
    <div class="rr">
      <div class="hero">
        <svg class="pole" viewBox="0 0 120 220" role="img" aria-label="{flag} flag">
          <rect x="14" y="8" width="6" height="210" rx="3" fill="#C9CCC6"/>
          <circle cx="17" cy="8" r="7" fill="#F2C230"/>
          <path class="flag {flag}" d="M20 20 C 50 10, 70 34, 112 22 L 112 82 C 70 94, 50 70, 20 80 Z"/>
        </svg>
        <div>
          <p class="day">Evening forecast for {day}</p>
          <h2 class="call {flag}">{call}</h2>
          <p class="why">{why}</p>
        </div>
      </div>
      <div class="court">{squares}</div>
      <p class="truth">{truth} <strong>{verdict}</strong></p>
    </div>"""

CSS = """
.gradio-container { background: #2E3236 !important; font-family: 'Atkinson Hyperlegible', system-ui, sans-serif !important; }
.gradio-container h1, .rr h2 { font-family: 'Bricolage Grotesque', system-ui, sans-serif; }
#title h1 { color: #F4F4EF; font-size: 2.6rem; font-weight: 800; letter-spacing: -0.02em; margin: 0.2em 0 0; }
#title p { color: #C9CCC6; font-size: 1.1rem; max-width: 60ch; }
.rr, .rr p, .rr span, .rr strong, .rr div { color: #F4F4EF !important; }
.rr .sq.on, .rr .sq.on span { color: #22252A !important; }
.rr .day { color: #C9CCC6 !important; }
.rr .call.green { color: #7BD08F !important; } .rr .call.red { color: #FF8A73 !important; }
button.primary, button.primary:hover { background: #F2C230 !important; color: #22252A !important; border: none !important;
         font-family: 'Bricolage Grotesque', system-ui, sans-serif !important; font-size: 1.1rem !important; }
.hero { display: flex; gap: 1.5rem; align-items: center; margin: 0.5rem 0 1.5rem; }
.pole { width: 110px; flex: none; }
.flag { animation: wave 2.4s ease-in-out infinite; transform-origin: 20px 50px; }
.flag.green { fill: #3E9B57; } .flag.red { fill: #D6402B; }
@keyframes wave { 50% { transform: skewY(-3deg) scaleX(0.97); } }
@media (prefers-reduced-motion: reduce) { .flag { animation: none; } }
.day { color: #C9CCC6; margin: 0; font-size: 1rem; }
.call { font-size: 2.8rem; line-height: 1.05; margin: 0.15em 0; font-weight: 800; }
.call.green { color: #7BD08F; } .call.red { color: #FF8A73; }
.why { font-size: 1.15rem; margin: 0; }
.court { display: grid; grid-template-columns: repeat(3, 1fr); border: 4px solid #F4F4EF;
         max-width: 560px; }
.sq { border: 2px solid #F4F4EF; padding: 0.9rem 0.6rem; min-height: 96px; display: flex;
      flex-direction: column; gap: 0.25rem; }
.sq.off { opacity: 0.55; }
.sq.on { background: #F2C230; color: #22252A; opacity: 1; }
.sq .ic { font-size: 1.6rem; } .sq .nm { font-weight: 700; }
.sq .real { font-size: 0.8rem; border: 1.5px solid currentColor; border-radius: 99px;
            padding: 0 0.5em; align-self: flex-start; }
.truth { margin-top: 1.2rem; font-size: 1.05rem; max-width: 60ch; }
#record { border-top: 2px dashed #5A6066; padding-top: 0.8rem; }
#record p { color: #C9CCC6 !important; font-size: 1.05rem; }
#record strong { color: #F4F4EF !important; }
#note p { color: #9DA3A8 !important; font-size: 0.9rem; }
#title p { color: #C9CCC6 !important; }
@media (max-width: 600px) { .hero { flex-direction: column; align-items: flex-start; } .call { font-size: 2.1rem; } }
"""

HEAD = ('<link rel="preconnect" href="https://fonts.googleapis.com">'
        '<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:wght@400;700'
        '&family=Bricolage+Grotesque:opsz,wght@12..96,800&display=swap" rel="stylesheet">')

with gr.Blocks(title="Recess Ready: Philadelphia") as demo:
    gr.Markdown("# Recess Ready\nIs tomorrow safe for outdoor recess at Philadelphia schools? "
                "Pick a school day to see what Recess Ready would have told your school the evening before.",
                elem_id="title")
    with gr.Row():
        date_box = gr.Dropdown(test_dates, value=test_dates[0], label="School day (2025-26 test year)", scale=3)
        go = gr.Button("Check this day", variant="primary", scale=1)
    result = gr.HTML(recess_check(test_dates[0]))
    gr.Markdown(f"**How good is it?** On the {total_unsafe} unsafe school days in our test year, "
                f"Recess Ready warned about **{model_caught}**. "
                f'Just guessing "tomorrow = today" warned about **{base_caught}**.', elem_id="record")
    gr.Markdown("Prototype, not official guidance. Always follow National Weather Service alerts and your school district. "
                "Air data is a computer-model estimate (CAMS via Open-Meteo), not a sensor reading.", elem_id="note")
    go.click(recess_check, inputs=date_box, outputs=result)
    date_box.change(recess_check, inputs=date_box, outputs=result)

demo.launch(css=CSS, head=HEAD)